# Только ASR → один podcast.json

Диаризацию не гоняем. Дописываем пустые тексты и собираем JSON.

1. Runtime → **T4 GPU**
2. Files: залей `speakers (1).zip`, `diarize_podcast.py`, `build_transcript_json.py`

In [ ]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), "Runtime → Change runtime type → T4 GPU, потом Restart"
print("CUDA", torch.cuda.get_device_name(0))

In [ ]:
!pip -q install transformers huggingface_hub soundfile sentencepiece hydra-core omegaconf
import torch
print("CUDA after install", torch.cuda.is_available())

In [ ]:
from pathlib import Path
import zipfile, shutil

src = next((p for p in [
    Path("speakers (1).zip"), Path("speakers.zip"), Path("/content/speakers (1).zip"), Path("/content/speakers.zip")
] if p.exists()), None)
assert src is not None, "Залей speakers (1).zip в Files слева"
assert Path("diarize_podcast.py").exists() and Path("build_transcript_json.py").exists(), "Залей diarize_podcast.py и build_transcript_json.py"

out = Path("speakers_run")
if out.exists():
    shutil.rmtree(out)
out.mkdir()
with zipfile.ZipFile(src) as z:
    z.extractall(out)
# zip may contain SPEAKER_* at root or one level down
root = out
if not any(out.glob("SPEAKER_*")):
    kids = [p for p in out.iterdir() if p.is_dir()]
    if len(kids) == 1:
        root = kids[0]
print("folder", root)
print("clips", len(list(root.glob("SPEAKER_*/*.wav"))))
open("ASR_ROOT.txt", "w").write(str(root))

In [ ]:
from pathlib import Path
import shutil, subprocess, sys

root = Path(open("ASR_ROOT.txt").read().strip())
print("ASR only, skip pyannote:", root)
subprocess.check_call([
    sys.executable, "-u", "build_transcript_json.py", str(root),
    "--gigaam-variant", "large_ctc",
])
shutil.copy(root / "podcast.json", "podcast.json")
shutil.make_archive("speakers_asr", "zip", root)
print("DONE — скачай podcast.json и speakers_asr.zip")